# Task 2.1 — Estrategias de generación

## Imports y reproducibilidad

In [1]:
import math
import os
import random

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 1337
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
if torch.backends.mps.is_available():
    torch.mps.manual_seed(SEED)

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print("device:", device)

device: mps


## Modelo

In [2]:
import math
import time

import matplotlib.pyplot as plt
import torch.nn as nn
import torch.nn.functional as F


def layer_norm(x, gamma, beta, eps=1e-5):
    mean = x.mean(dim=-1, keepdim=True)
    var = x.var(dim=-1, unbiased=False, keepdim=True)
    return (x - mean) / torch.sqrt(var + eps) * gamma + beta


def make_positional_encoding(max_len, d_model):
    pe = torch.zeros(max_len, d_model)
    pos = torch.arange(max_len).unsqueeze(1).float()
    div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(pos * div)
    pe[:, 1::2] = torch.cos(pos * div)
    return pe


class Block(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        self.WQ = nn.Parameter(torch.randn(d_model, d_model) * 0.02)
        self.WK = nn.Parameter(torch.randn(d_model, d_model) * 0.02)
        self.WV = nn.Parameter(torch.randn(d_model, d_model) * 0.02)
        self.WO = nn.Parameter(torch.randn(d_model, d_model) * 0.02)

        self.gamma1 = nn.Parameter(torch.ones(d_model))
        self.beta1 = nn.Parameter(torch.zeros(d_model))

        self.W1 = nn.Parameter(torch.randn(d_model, d_ff) * 0.02)
        self.bias1 = nn.Parameter(torch.zeros(d_ff))
        self.W2 = nn.Parameter(torch.randn(d_ff, d_model) * 0.02)
        self.bias2 = nn.Parameter(torch.zeros(d_model))

        self.gamma2 = nn.Parameter(torch.ones(d_model))
        self.beta2 = nn.Parameter(torch.zeros(d_model))

    def masked_self_attention(self, x, causal_mask):
        B, T, C = x.shape
        Q = (x @ self.WQ).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        K = (x @ self.WK).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        V = (x @ self.WV).view(B, T, self.n_heads, self.d_k).transpose(1, 2)

        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_k)  # (B, n_heads, T, T)
        scores = scores.masked_fill(causal_mask[:T, :T], float("-inf"))
        attn_w = F.softmax(scores, dim=-1)

        out = (attn_w @ V).transpose(1, 2).contiguous().view(B, T, C)
        return out @ self.WO

    def feed_forward(self, x):
        h = F.relu(x @ self.W1 + self.bias1)
        return h @ self.W2 + self.bias2

    def forward(self, x, causal_mask):
        # pre-LayerNorm (estilo GPT-2): estabiliza el entrenamiento al apilar capas
        x = x + self.masked_self_attention(layer_norm(x, self.gamma1, self.beta1), causal_mask)
        x = x + self.feed_forward(layer_norm(x, self.gamma2, self.beta2))
        return x


class MiniGPT(nn.Module):
    def __init__(self, V, d_model, n_heads, n_layers, block_size):
        super().__init__()
        self.E = nn.Parameter(torch.randn(V, d_model) * 0.02)
        self.register_buffer("PE", make_positional_encoding(block_size, d_model))
        self.register_buffer("causal_mask", torch.triu(torch.ones(block_size, block_size), diagonal=1).bool())

        self.blocks = nn.ModuleList([Block(d_model, n_heads, 4 * d_model) for _ in range(n_layers)])

        self.gamma_f = nn.Parameter(torch.ones(d_model))
        self.beta_f = nn.Parameter(torch.zeros(d_model))
        self.Wlm = nn.Parameter(torch.randn(d_model, V) * 0.02)
        self.blm = nn.Parameter(torch.zeros(V))

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.E[idx] + self.PE[:T]
        for block in self.blocks:
            x = block(x, self.causal_mask)
        x = layer_norm(x, self.gamma_f, self.beta_f)
        logits = x @ self.Wlm + self.blm  # (B, T, V)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

## Carga del checkpoint

In [3]:
CKPT_PATH = os.path.join("checkpoints", "mini_gpt.pt")
if not os.path.exists(CKPT_PATH):
    raise FileNotFoundError(f"No existe {CKPT_PATH}: ejecute task1.ipynb primero para entrenar el modelo.")

ckpt = torch.load(CKPT_PATH, map_location="cpu")
stoi, itos, config = ckpt["stoi"], ckpt["itos"], ckpt["config"]
block_size = config["block_size"]
vocab_size = len(stoi)

def encode(s):
    return [stoi[c] for c in s]

def decode(ids):
    return "".join(itos[int(i)] for i in ids)

model = MiniGPT(vocab_size, **config).to(device)
model.load_state_dict(ckpt["model"])
model.eval()

val_loss = ckpt["history"]["val"][-1]
print("config:", config, "| vocab_size:", vocab_size)
print(f"val loss: {val_loss:.4f} | perplejidad: {math.exp(val_loss):.4f}")

config: {'block_size': 128, 'd_model': 128, 'n_heads': 4, 'n_layers': 3} | vocab_size: 65
val loss: 1.9682 | perplejidad: 7.1580


## Estrategias de muestreo

In [4]:
def greedy(logits):
    return int(torch.argmax(logits))


def top_k(logits, k, tau=1.0):
    values, indices = torch.topk(logits / tau, k)
    probs = F.softmax(values, dim=-1)
    return int(indices[torch.multinomial(probs, num_samples=1)])


def top_p(logits, p, tau=1.0):
    probs = F.softmax(logits / tau, dim=-1)
    sorted_probs, indices = torch.sort(probs, descending=True)
    cumulative = torch.cumsum(sorted_probs, dim=-1)
    # conjunto mínimo cuya probabilidad acumulada alcanza p (siempre incluye el más probable)
    keep = (cumulative - sorted_probs) < p
    kept_probs = sorted_probs[keep] / sorted_probs[keep].sum()
    return int(indices[keep][torch.multinomial(kept_probs, num_samples=1)])

## Función generate

In [5]:
STRATEGIES = {"greedy": greedy, "top-k": top_k, "top-p": top_p}


@torch.no_grad()
def generate(prompt, max_new_tokens, strategy, **kwargs):
    if strategy not in STRATEGIES:
        raise ValueError(f"Estrategia desconocida: {strategy!r}. Opciones: {list(STRATEGIES)}")
    pick = STRATEGIES[strategy]

    model.eval()
    ids = encode(prompt)
    for _ in range(max_new_tokens):
        context = torch.tensor([ids[-block_size:]], dtype=torch.long, device=device)
        logits, _ = model(context)
        ids.append(pick(logits[0, -1].cpu(), **kwargs))
    return decode(ids)

## Configuraciones

In [6]:
PROMPT = "ROMEO:"
N_SAMPLES = 5
MAX_NEW_TOKENS = 200

CONFIGS = {
    "A": ("greedy", {}),
    "B": ("top-k", dict(k=5, tau=0.7)),
    "C": ("top-k", dict(k=50, tau=1.0)),
    "D": ("top-p", dict(p=0.70, tau=0.7)),
    "E": ("top-p", dict(p=0.95, tau=1.0)),
}

## Muestras

In [7]:
torch.manual_seed(SEED)
samples = {}
for name, (strategy, kwargs) in CONFIGS.items():
    samples[name] = [generate(PROMPT, MAX_NEW_TOKENS, strategy, **kwargs) for _ in range(N_SAMPLES)]
    print(f"=== {name} · {strategy} · {kwargs} ===")
    for i, text in enumerate(samples[name], 1):
        print(f"--- muestra {i} ---")
        print(text)
        print()

=== A · greedy · {} ===
--- muestra 1 ---
ROMEO:
I what the the the the the the the the the the the the the the
To the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the t

--- muestra 2 ---
ROMEO:
I what the the the the the the the the the the the the the the
To the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the t

--- muestra 3 ---
ROMEO:
I what the the the the the the the the the the the the the the
To the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the t

--- muestra 4 ---
ROMEO:
I what the the the the the the the the the the the the the the
To the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the t

--- muestra 5 ---
ROMEO:
I what the the the the the the the the the the 

=== B · top-k · {'k': 5, 'tau': 0.7} ===
--- muestra 1 ---
ROMEO:
What thou mis to were stalle would sernones
In hear here make will will will me with hencand.

LOREO:
I were have a the stoo and the with share me,
Tho hear hond tree thou with worse to see
The the m

--- muestra 2 ---
ROMEO:
I'll to woor that shall the we were make to thing
The hould to him, the should the thou sir a she
Whath will tay alle sir the alt souss a sin.

LAUSES:
The me that stour on the and sto the stay.

MUL

--- muestra 3 ---
ROMEO:
Ay that the soun soul sour the a shume
Whe here shous thing hance an his me the hone
And stise to me stongs angert and streath
The and with whou see to angers, this with with and worse.

Serive, then

--- muestra 4 ---
ROMEO:
What make to him the to maress,
We may the have thou wart to the thich was shand
Than a stall the that there are this then mise
To shane strere ont thou with and me my tay.

She work, seet and to thi

--- muestra 5 ---
ROMEO:
I with marry the that would an

=== C · top-k · {'k': 50, 'tau': 1.0} ===
--- muestra 1 ---
ROMEO:
I'll of is hear dentle made,--But thour,
Anvaice lain; and
Then upememen loody tecown theamight
And coutin's am here shicky crawn notiomely
An chorat as mery I counsled it aganst.
Tha mangur sterer; 

--- muestra 2 ---
ROMEO:
Thr-me the hawshord no is the bepriegatiome.

ENEO:
Nome, t to ofath anter, a shy litgornt, For dord?

HERCUCERY:
Doo mosthan ploing cor' thou RGjo:
That to's sirser, may will ancous; wheat that
Mare

--- muestra 3 ---
ROMEO:
Sen it prilad contelt to doh that timinis.

MERUTCKEs--revers
Thance bothere hour to Trist not wele,
Hand pance almenout how and them es&ashtlerss,
Prilistior
Thegu it thid meands of, cleast same pea

--- muestra 4 ---
ROMEO:
The cousey day'd the my beg heinge?

MAMORDTEO:
I tnemowerd, At mariows:-houary, 'sin?

MEENEO:

C'et morer:
Is fay my food sive, but see and you I nouve;

Suir a spution ort good lave; but ent dove 

--- muestra 5 ---
ROMEO:
Mith on couse may mich will,


=== D · top-p · {'p': 0.7, 'tau': 0.7} ===
--- muestra 1 ---
ROMEO:
That some have be to the with strown,
What the are the a coust here the faintent
The that with him me that in for in that a me.

MAULINCE:
I with me be stay a him seat on the me a mont were,
And the 

--- muestra 2 ---
ROMEO:
What be word beaing the brother him the with
Whe have me make of soul man will and the brow
And the the will with a do in on the seat with the my mand.

Fir will the in be the did with word that in m

--- muestra 3 ---
ROMEO:
I she that the shall be stay sour that my thee
are his the to his that with and his that my be hand
I more worse that the causter hear hears
Whe bean on of the that stand and with thee wear of a than

--- muestra 4 ---
ROMEO:
That be it that with may hath the the sour the to the and a the
To mane of the with the with with be more,
And staing one an the in the the be a the be then the with but him in and a have
That he hav

--- muestra 5 ---
ROMEO:
Whou me me my that shall sou

=== E · top-p · {'p': 0.95, 'tau': 1.0} ===
--- muestra 1 ---
ROMEO:
Fare whould benign: thou paur toe!

CLANET:
For his exing mut the hord reforectin.
Fill flathers on out nemast shey dim you it our shis a sar,
I novent not acains int of you do greatuo,
Hell mandance

--- muestra 2 ---
ROMEO:
I sexplie upoumpast freant a farteron;
That in hould end flowe beay in no gemmimin.

MUKENTY REO:
Than blingens, that that him lamithed I feem and
Ertricht knour o'll the no anse of as it meand
If ab

--- muestra 3 ---
ROMEO:
Goright kidin. Bonoly! for cars winksing in to frice;
Tindely fim on of lane with like bage
To bely you sigh mysalst; that by diy.

PALEO:
By raccatar, now.

RIS:
Poir, my good weart minsen cont agas

--- muestra 4 ---
ROMEO:
A be comblint lave havind me, where feep.
'Th let rabiant his embuctie trang tissul
LOM:
Fay worster macth sill to ancers may alrear
graty's hig hearen brainst coin o' brownoss
I her than sure to upp

--- muestra 5 ---
ROMEO:
Pew sway matt didio.

QUEEN

## Verificación

In [8]:
for name, texts in samples.items():
    assert len(texts) == N_SAMPLES
    for text in texts:
        assert text.startswith(PROMPT)
        assert len(text) == len(PROMPT) + MAX_NEW_TOKENS

assert len(set(samples["A"])) == 1
assert generate(PROMPT, 50, "top-k", k=1, tau=1.0) == generate(PROMPT, 50, "greedy")
print("OK")

OK
